# 03 — Domain-Specific Intent Taxonomy Formulation (@AppleSupport)

## Executive Summary & Core Objective
In an enterprise customer support pipeline, **intent classification** is the primary decision gate. It determines:
1. **Context & Retrieval Partitioning**: Which historical case cluster to search.
2. **Evidence-Guided Prompting**: Which policy constraints and few-shot exemplars to inject.
3. **Operational Routing**: Whether an inquiry is safe for autonomous fulfillment (`AUTO_HANDLE`) or requires human intervention (`HUMAN_ESCALATION`).

This notebook formalizes the **7-class Mutually Exclusive, Collectively Exhaustive (MECE)** intent taxonomy derived from real `@AppleSupport` dialogues in the Kaggle *Customer Support on Twitter* corpus.

In [ ]:
import json
import re
import sys
from collections import Counter
from pathlib import Path
import pandas as pd

# Ensure repository root is on sys.path
REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.models.domain_models import SupportIntent, INTENT_LABELS, RoutingDecision
from app.models.taxonomy_models import IntentTaxonomySchema
from scripts.data.classify_intents import classify_text_intent, load_intent_taxonomy

print("Environment initialized successfully. PyTorch/Pandas available.")

## 1. Dataset Ingestion & Customer Inquiry Inspection
We load the reconstructed `@AppleSupport` conversation sample (`data/processed/applesupport_sample.jsonl`) generated in Phase 3.

In [ ]:
data_path = REPO_ROOT / "data" / "processed" / "applesupport_sample.jsonl"
assert data_path.exists(), f"Missing data file: {data_path}"

conversations = []
with open(data_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            conversations.append(json.loads(line))

print(f"Loaded {len(conversations)} verified conversation threads.")
df = pd.DataFrame(conversations)
df[["conversation_id", "first_inquiry", "brand", "turn_count", "status"]].head(5)

## 2. Empirical N-Gram & Frequency Analysis
We analyze customer opening messages to observe high-frequency keywords, technical identifiers, and problem signals.

In [ ]:
def extract_ngrams(texts, n=2):
    ngrams = []
    for t in texts:
        words = re.findall(r"\b[a-zA-Z]{3,}\b", t.lower())
        for i in range(len(words) - n + 1):
            ngrams.append(" ".join(words[i:i+n]))
    return Counter(ngrams)

inquiries = [c.get("first_inquiry", "") for c in conversations]
bigrams = extract_ngrams(inquiries, n=2)

print("Top 15 Most Common Bigrams in Customer Inquiries:")
for bg, cnt in bigrams.most_common(15):
    print(f"  {bg:<24}: {cnt}")

## 3. The 7-Class MECE Intent Taxonomy
Based on domain clustering and support operations, we establish 7 mutually exclusive classes:

| Intent Code | Display Label | Scope | Default Routing | Risk Level |
| :--- | :--- | :--- | :---: | :---: |
| `OPERATING_SYSTEM_UPDATES` | OS & iOS Updates | iOS/macOS version glitches, update verification, autocorrect bugs | `AUTO_HANDLE` | LOW |
| `BATTERY_POWER_HARDWARE` | Battery & Hardware | Battery drain, thermal overheating, cracked screens, charging port | `HUMAN_ESCALATION` | CRITICAL |
| `ACCOUNT_APPLE_ID` | Apple ID & Account | 2FA lockouts, password resets, Apple ID region changes | `HUMAN_ESCALATION` | HIGH |
| `CONNECTIVITY_NETWORKING` | Connectivity & Wi-Fi | Wi-Fi disconnects, Bluetooth pairing, cellular / SIM errors | `AUTO_HANDLE` | LOW |
| `AUDIO_ACCESSORIES` | Audio & Accessories | AirPods, headphone jack adapter, crackling speaker, microphone | `AUTO_HANDLE` | LOW |
| `SUBSCRIPTIONS_BILLING` | Billing & Subscriptions | Unauthorized charges, iTunes refunds, subscription renewals | `HUMAN_ESCALATION` | CRITICAL |
| `GENERAL_INQUIRY` | General Support | Store hours, reservations, pre-orders, ambiguous inquiries | `HUMAN_ESCALATION` | MEDIUM |

Let us validate the formal schema.

In [ ]:
taxonomy_file = REPO_ROOT / "data" / "taxonomy" / "intent_taxonomy.json"
schema = load_intent_taxonomy(taxonomy_file)

print(f"Taxonomy Version: {schema.version}")
print(f"Target Brand: {schema.brand}")
print(f"Total Classes: {schema.num_classes}")
print(f"MECE Verified: {schema.metadata.get('mece_verified')}")

for cls in schema.classes:
    print(f"\n[{cls.code.value}] -> {cls.label}")
    print(f"  Routing: {cls.default_routing.value} | Risk: {cls.risk_level.value} | Priority: {cls.priority_rank}")
    print(f"  Signals: {', '.join(cls.key_signals[:5])}...")

## 4. Benchmark Canonical Intent Verification
We evaluate canonical queries across each of the 7 classes to ensure the classifier captures the domain semantics with high confidence.

In [ ]:
benchmark_queries = [
    ("Stuck on 'Verifying update' for iOS 11.0.3, rebooting didn't help.", SupportIntent.OPERATING_SYSTEM_UPDATES),
    ("My iPhone 7 battery percentage drops from 80% to 15% and back gets burning hot.", SupportIntent.BATTERY_POWER_HARDWARE),
    ("Locked out of my Apple ID because I changed my mobile number and cannot get 2FA code.", SupportIntent.ACCOUNT_APPLE_ID),
    ("Why does my Wi-Fi keep disconnecting every time my phone locks on cellular/home network?", SupportIntent.CONNECTIVITY_NETWORKING),
    ("My right AirPod won't connect or charge in the case, audio crackles.", SupportIntent.AUDIO_ACCESSORIES),
    ("I was charged $9.99 from itunes.com/bill for a subscription I cancelled, need a refund.", SupportIntent.SUBSCRIPTIONS_BILLING),
    ("What are the opening hours for the Apple Store in Covent Garden on Sunday?", SupportIntent.GENERAL_INQUIRY),
]

print(f"{'Expected Intent':<28} | {'Predicted Intent':<28} | {'Conf':<5} | {'Signals'}")
print("-" * 85)
for query, expected in benchmark_queries:
    pred_enum, conf, signals = classify_text_intent(query)
    status = "PASS" if pred_enum == expected else "FAIL"
    print(f"{INTENT_LABELS[expected]:<28} | {INTENT_LABELS[pred_enum]:<28} | {conf:<5.2f} | {signals[:2]} [{status}]")
    assert pred_enum == expected, f"Mismatch: expected {expected}, got {pred_enum}"

## 5. Disambiguation & Priority Hierarchy Validation
When customer inquiries span multiple intent domains, deterministic disambiguation rules ensure safety and financial correctness:
- **Safety / Swollen Battery** overrides Software Updates (`BATTERY_POWER_HARDWARE` > `OPERATING_SYSTEM_UPDATES`)
- **Financial / Unauthorized Charge** overrides General Inquiries (`SUBSCRIPTIONS_BILLING` > `GENERAL_INQUIRY`)
- **Audio Accessories** overrides generic Bluetooth (`AUDIO_ACCESSORIES` > `CONNECTIVITY_NETWORKING`)
- **Account / 2FA Lockout** overrides App Store Update downloads (`ACCOUNT_APPLE_ID` > `OPERATING_SYSTEM_UPDATES`)

In [ ]:
conflict_cases = [
    (
        "After installing the iOS 11 update my iPhone battery became swollen and pushed the screen out.",
        SupportIntent.BATTERY_POWER_HARDWARE,
        "Safety hazard priority"
    ),
    (
        "I checked my receipt and noticed an unauthorized subscription charge from iTunes billing, need refund.",
        SupportIntent.SUBSCRIPTIONS_BILLING,
        "Financial liability priority"
    ),
    (
        "My AirPods won't connect via Bluetooth to my phone.",
        SupportIntent.AUDIO_ACCESSORIES,
        "Peripheral specificity over general networking"
    ),
    (
        "I cannot update any apps because my Apple ID has been disabled for security reasons.",
        SupportIntent.ACCOUNT_APPLE_ID,
        "Authentication lockout over OS updates"
    ),
]

for text, expected, rationale in conflict_cases:
    intent, conf, signals = classify_text_intent(text)
    print(f"Query: '{text[:65]}...'")
    print(f"  Result: {INTENT_LABELS[intent]} (Confidence: {conf:.2f})")
    print(f"  Rationale: {rationale}")
    assert intent == expected, f"Tie-break failure: expected {expected}, got {intent}"
    print("  -> Correctly Disambiguated!\n")

## 6. Empirical Distribution Across Preprocessed Data
We inspect the distribution of classified intents across the `@AppleSupport` conversation sample.

In [ ]:
dist_file = REPO_ROOT / "experiments" / "intent_distribution.json"
if dist_file.exists():
    with open(dist_file, "r", encoding="utf-8") as f:
        dist_data = json.load(f)
    print("Empirical Distribution Summary:")
    stats_df = pd.DataFrame(dist_data.get("detailed_statistics", {})).T
    print(stats_df[["count", "percentage", "avg_confidence"]])
else:
    counts = Counter(c.get("intent") for c in conversations)
    for intent_name, count in counts.most_common():
        pct = (count / len(conversations)) * 100
        print(f"  {intent_name:<26}: {count:>3} ({pct:.1f}%)")

## 7. Conclusion & Handoff to Phase 5
With the 7-class intent taxonomy formalized, tested, and validated:
1. **Golden Set Curation (Phase 5)** has precise boundary definitions and disambiguation rubrics.
2. **Baseline and Final Intent Models (Phase 6)** have an unambiguous target space.
3. **Historical Retrieval (Phase 7)** can partition candidate search by high-level domain category.